In [1]:
import pandas as pd
import numpy as np

data_dir = '/Users/gabrielarm300/cs197_fall2026/physionet.org/files/eicu-crd/2.0/'

In [2]:
patient = pd.read_csv(data_dir + 'patient.csv.gz')
hospital = pd.read_csv(data_dir + 'hospital.csv.gz')
diagnosis = pd.read_csv(data_dir + 'diagnosis.csv.gz')
print(patient.shape)

(200859, 29)


In [3]:
pd.read_csv(data_dir + 'vitalPeriodic.csv.gz', nrows=5)

,vitalperiodicid,patientunitstayid,observationoffset,temperature,sao2,heartrate,respiration,cvp,etco2,systemicsystolic,systemicdiastolic,systemicmean,pasystolic,padiastolic,pamean,st1,st2,st3,icp
0,37376747,141168,2059,NaN,NaN,92,NaN,30.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,37404957,141168,1289,NaN,NaN,118,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,37385871,141168,1794,NaN,91.0,78,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,37401664,141168,1374,NaN,90.0,118,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,37377404,141168,2039,NaN,98.0,92,NaN,33.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
# same cohort filters as the homework
patient = patient[patient.hospitaldischargeoffset >= 24*60]      # stayed at least 24h
patient = patient[~(-patient.hospitaladmitoffset > 48*60)]       # ICU started within first 48h
n_cohort = patient.patienthealthsystemstayid.nunique()
print(patient.shape, n_cohort)

(147487, 29) 135778


In [5]:
# age as a number ('> 89' becomes 90, like the homework)
patient['age_num'] = pd.to_numeric(patient.age.replace('> 89', '90'), errors='coerce')

In [6]:
# ICU stay start/end, in hours since hospital admission
patient['start_h'] = -patient.hospitaladmitoffset / 60
patient['end_h']   = patient.start_h + patient.unitdischargeoffset / 60

In [7]:
# one row per hospitalization
grp = patient.sort_values('start_h').groupby('patienthealthsystemstayid')
hosp_df = pd.DataFrame({
    'hospitalid': grp.hospitalid.first(), #the .first() is for people with the same hospitalization but different ICU stays
    'age':        grp.age_num.first(),
    'gender':     grp.gender.first(),
    'ethnicity':  grp.ethnicity.first(),
    'start_h':    grp.start_h.min(), #earliest
    'end_h':      grp.end_h.max(), #latest
    'died_hosp':  grp.hospitaldischargestatus.apply(lambda s: (s == 'Expired').any()),
    'died_icu':   grp.unitdischargestatus.apply(lambda s: (s == 'Expired').any()),
})
hosp_df = (hosp_df.reset_index() # the ID isn't the front row (identifying), but just one row
           .merge(hospital[['hospitalid', 'region', 'numbedscategory']], on='hospitalid', how='left')
           .set_index('patienthealthsystemstayid'))

In [8]:
# hours observed in each half-day window (to compare how people have improved/gotten worse every 12 hors)
edges = [0, 12, 24, 36, 48]
for i in range(4):
    lo = np.maximum(hosp_df.start_h, edges[i])
    hi = np.minimum(hosp_df.end_h,   edges[i+1])
    hosp_df[f'hrs_h{i+1}'] = (hi - lo).clip(lower=0)

In [9]:
hosp_df.to_parquet('step1_hosp.parquet') #Saves the table to a file so you never have to rebuild it? I honeslty don't know how this works but sounds helpful

In [10]:
keep = patient[['patientunitstayid', 'patienthealthsystemstayid', 'hospitaladmitoffset']]
cols = ['patientunitstayid', 'labname', 'labresultoffset', 'labresult'] #who, which test, when, and the value

draws, cr_rows = [], []
for chunk in pd.read_csv(data_dir + 'lab.csv.gz', usecols=cols, chunksize=2_000_000): # 2M rows at a time because there are too many
    chunk = chunk.merge(keep, on='patientunitstayid')  #drop whatever has missing data on keep       
    chunk['hours'] = (-chunk.hospitaladmitoffset + chunk.labresultoffset) / 60   # hours since hospital admit
    chunk = chunk[(chunk.hours >= 0) & (chunk.hours < 48)]
    is_cr = chunk.labname.str.lower().eq('creatinine') #checks if the lab was for creatine

    cr_rows.append(chunk[is_cr][['patienthealthsystemstayid', 'hours', 'labresult']])
    draws.append(chunk[~is_cr][['patienthealthsystemstayid', 'hours']].drop_duplicates())

lab_draws = pd.concat(draws).drop_duplicates() #how often was the patient checked
cr = pd.concat(cr_rows) #glue chunks together

lab_draws.to_pickle('step2_lab_draws.pkl') #saving
cr.to_pickle('step2_creatinine.pkl') #saving

In [11]:
aper = []
for chunk in pd.read_csv(data_dir + 'vitalAperiodic.csv.gz', chunksize=2_000_000):
    # drop rows where every measurement is blank (nothing was actually recorded)
    measurements = chunk.columns.difference(['vitalaperiodicid', 'patientunitstayid', 'observationoffset']) #don't include these rows
    chunk = chunk[chunk[measurements].notna().any(axis=1)]

    chunk = chunk.merge(keep, on='patientunitstayid')
    chunk['hours'] = (-chunk.hospitaladmitoffset + chunk.observationoffset) / 60
    chunk = chunk[(chunk.hours >= 0) & (chunk.hours < 48)]
    aper.append(chunk[['patienthealthsystemstayid', 'hours']].drop_duplicates())

aper_ev = pd.concat(aper).drop_duplicates() #glue chunks together
aper_ev.to_pickle('step3_aper_events.pkl')

In [12]:
edges = [0, 12, 24, 36, 48]
MIN_HRS = 2          # need at least 2 observed hours in a window to compute a rate

def count_per_window(ev, name):
    ev = ev.copy()
    ev['win'] = pd.cut(ev.hours, bins=edges, labels=['h1', 'h2', 'h3', 'h4'], right=False)
    c = (ev.groupby(['patienthealthsystemstayid', 'win'], observed=False).size()
           .unstack(fill_value=0))
    c.columns = [f'{name}_n_{w}' for w in c.columns]
    return c

feats = hosp_df.copy()

for ev, name in [(lab_draws, 'lab'), (aper_ev, 'aper')]:
    c = count_per_window(ev, name)
    feats = feats.join(c)
    feats[c.columns] = feats[c.columns].fillna(0)      # no events = 0 counts

# creatinine check counts (separate "kidney monitoring" feature)
c = count_per_window(cr[['patienthealthsystemstayid', 'hours']].drop_duplicates(), 'cr')
feats = feats.join(c)
feats[c.columns] = feats[c.columns].fillna(0)

# flag: any aperiodic data at all ("never recorded" vs. "never measured")
feats['has_aper'] = feats.index.isin(aper_ev.patienthealthsystemstayid)

# rates = count / hours observed; blank if under MIN_HRS hours observed
for name in ['lab', 'aper', 'cr']:
    for w in ['h1', 'h2', 'h3', 'h4']:
        hrs = feats[f'hrs_{w}'].where(feats[f'hrs_{w}'] >= MIN_HRS)
        feats[f'{name}_rate_{w}'] = feats[f'{name}_n_{w}'] / hrs

feats.to_pickle('step4_feats.pkl')

In [13]:
cr_s = cr.sort_values('hours')
d1 = cr_s[cr_s.hours < 24].groupby('patienthealthsystemstayid').labresult.last().rename('cr_d1')
d2 = cr_s[(cr_s.hours >= 24) & (cr_s.hours < 48)].groupby('patienthealthsystemstayid').labresult.last().rename('cr_d2')
cr_first = cr_s[cr_s.hours < 24].groupby('patienthealthsystemstayid').labresult.first().rename('cr_first')

out = feats.join([d1, d2, cr_first])
out['cr_pct'] = (out.cr_d2 - out.cr_d1) / out.cr_d1
out['cr_abs'] = out.cr_d2 - out.cr_d1

PCT = 0.25
has_both = out.cr_d1.notna() & out.cr_d2.notna()
no_d2    = out.cr_d1.notna() & out.cr_d2.isna()
died     = out.died_hosp.astype(bool)

conditions = [
    has_both & (out.cr_pct >= PCT),
    has_both & (out.cr_pct <= -PCT),
    has_both,
    no_d2 & died,
    no_d2 & ~died,
]
choices = ['worsened', 'improved', 'stable', 'died', 'no_day2_alive']

out['label'] = np.select(conditions, choices, default='no_baseline_cr')

out.to_pickle('step5_outcome.pkl')

In [14]:
keep = patient[['patientunitstayid', 'patienthealthsystemstayid', 'hospitaladmitoffset']]

# ---- kidney diagnosis in the first 24h ----
dx = pd.read_csv(data_dir + 'diagnosis.csv.gz', usecols=['patientunitstayid', 'diagnosisoffset', 'icd9code'])
dx = dx.merge(keep, on='patientunitstayid')
dx['hours'] = (-dx.hospitaladmitoffset + dx.diagnosisoffset) / 60
is_kid = dx.icd9code.astype(str).str.contains(r'\bN17|\bN18|\b584|\b585', regex=True)
kidney_dx_ids = set(dx[is_kid & (dx.hours <= 24)].patienthealthsystemstayid)

# ---- dialysis ----
tr = pd.read_csv(data_dir + 'treatment.csv.gz', usecols=['patientunitstayid', 'treatmentoffset', 'treatmentstring'])
tr = tr[tr.treatmentstring.str.contains('dialysis', case=False, na=False)]
tr = tr.merge(keep, on='patientunitstayid')
tr['hours'] = (-tr.hospitaladmitoffset + tr.treatmentoffset) / 60
dial24_ids = set(tr[tr.hours < 24].patienthealthsystemstayid)
dial48_ids = set(tr[tr.hours < 48].patienthealthsystemstayid)

# ---- flags on the outcome table ----
out['kidney_dx'] = out.index.isin(kidney_dx_ids)
out['cr_high']   = out.cr_first > 1.5
out['kidney']    = out.kidney_dx | out.cr_high
out['dial_24']   = out.index.isin(dial24_ids)
out['dial_48']   = out.index.isin(dial48_ids)


kc = out[out.kidney & ~out.dial_48]

out.to_pickle('step6_cohort.pkl')

In [20]:
kc = out[out.kidney & ~out.dial_48].copy()

# day-1 rates: counts in 0-24h divided by observed hours; need at least 4 observed hours
hrs1 = (kc.hrs_h1 + kc.hrs_h2)
hrs1 = hrs1.where(hrs1 >= 4)
for name in ['lab', 'aper', 'cr']:
    kc[f'{name}_rate_d1'] = (kc[f'{name}_n_h1'] + kc[f'{name}_n_h2']) / hrs1
kc.loc[~kc.has_aper, 'aper_rate_d1'] = np.nan

# analysis sample: patients with a label on creatinine
d = kc[kc.label.isin(['improved', 'stable', 'worsened'])].copy()
d['y'] = (d.label == 'worsened').astype(int)
d['kidney_dx'] = d.kidney_dx.astype(int)
d['log_cr_d1'] = np.log(d.cr_d1.clip(lower=0.1))
for n in ['lab', 'aper', 'cr']:
    d[f'log_{n}_rate'] = np.log1p(d[f'{n}_rate_d1'])

print(len(d), d.y.mean().round(3))
print(d[['lab_rate_d1', 'aper_rate_d1', 'cr_rate_d1']].describe().round(2).to_string())

# ---- 1. plain look: worsened share by lab-rate quartile, within baseline creatinine bands ----
d['lab_q']  = pd.qcut(d.lab_rate_d1, 4, labels=['Q1 low', 'Q2', 'Q3', 'Q4 high'])
d['aper_q'] = pd.qcut(d.aper_rate_d1, 4, labels=['Q1 low', 'Q2', 'Q3', 'Q4 high'])
d['cr_band'] = pd.qcut(d.cr_d1, 3, labels=['low cr', 'mid cr', 'high cr'])

print(d.groupby('lab_q', observed=True).y.mean().round(3).to_string())
print(d.groupby(['cr_band', 'lab_q'], observed=True).y.mean().unstack().round(3).to_string())
print(d.groupby('aper_q', observed=True).y.mean().round(3).to_string())
print(d.groupby(['cr_band', 'aper_q'], observed=True).y.mean().unstack().round(3).to_string())

# ---- 2. models (a)-(d), tested on held-out hospitals ----
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.compose import make_column_transformer
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression

ctrl_num = ['log_cr_d1', 'age', 'kidney_dx']
ctrl_cat = ['gender', 'ethnicity', 'region', 'numbedscategory']
sets = {
    'a controls only': [],
    'b + lab rate':    ['log_lab_rate'],
    'c + aper rate':   ['log_aper_rate'],
    'd + both':        ['log_lab_rate', 'log_aper_rate'],
}

gkf = GroupKFold(n_splits=5)
for name, extra in sets.items():
    pre = make_column_transformer(
        (make_pipeline(SimpleImputer(strategy='median', add_indicator=True), StandardScaler()),
         ctrl_num + extra),
        (make_pipeline(SimpleImputer(strategy='constant', fill_value='unknown'),
                       OneHotEncoder(handle_unknown='ignore')), ctrl_cat))
    model = make_pipeline(pre, LogisticRegression(max_iter=2000))
    p = np.zeros(len(d))
    for tr_i, te_i in gkf.split(d, d.y, groups=d.hospitalid):
        model.fit(d.iloc[tr_i], d.y.iloc[tr_i])
        p[te_i] = model.predict_proba(d.iloc[te_i])[:, 1]
    print(f'{name:18s} AUROC {roc_auc_score(d.y, p):.3f}   AUPRC {average_precision_score(d.y, p):.3f}')
print('baseline AUPRC (random guessing):', d.y.mean().round(3))

d.to_pickle('step7_analysis.pkl')

27150 0.075
       lab_rate_d1  aper_rate_d1  cr_rate_d1
count     25354.00      24844.00    25354.00
mean          0.55          2.57        0.12
std           0.43          3.41        0.09
min           0.04          0.00        0.04
25%           0.27          1.27        0.05
50%           0.43          2.11        0.10
75%           0.68          3.36        0.15
max           5.77         59.89        1.38
lab_q
Q1 low     0.052
Q2         0.071
Q3         0.070
Q4 high    0.098
lab_q    Q1 low     Q2     Q3  Q4 high
cr_band                               
low cr    0.037  0.054  0.052    0.072
mid cr    0.048  0.078  0.080    0.117
high cr   0.075  0.082  0.076    0.107
aper_q
Q1 low     0.072
Q2         0.060
Q3         0.070
Q4 high    0.085
aper_q   Q1 low     Q2     Q3  Q4 high
cr_band                               
low cr    0.052  0.047  0.053    0.065
mid cr    0.081  0.060  0.075    0.098
high cr   0.090  0.076  0.082    0.090
a controls only    AUROC 0.552   AUPRC 0.089